# 05 — N-grams & Collocations

**Learning objective.** Model local word order with contiguous token sequences and inspect which phrases carry signal.

This notebook is intentionally **offline-reproducible**: the examples use local data or deterministic toy corpora so the rendered GitHub output can be trusted without hidden API calls. The focus is always **concept → inspectable representation → library implementation → result → failure modes → production implication**.

## 🧠 Visual engineering mental model

![Causal mindmap](assets/mindmaps/05_ngrams_collocations.svg)

Read the map **left → right**, then inspect the control knob above it. The learning goal is to predict how a control change propagates before running code.

## 🎛️ Change map — if you change this, what moves downstream?

| Change | Immediate effect | Downstream consequence |
|---|---|---|
| Increase word **n** | more local order is represented | feature count/sparsity explode |
| Use **character n-grams** | partial spelling becomes evidence | typo/morphology robustness improves |
| Increase `min_df` | rare n-grams disappear | model becomes smaller but rare phrases may be lost |

### Engineering rule
Change **one control at a time**, predict the direction of the effect, then measure whether reality matches the prediction.

## 🔮 Predict before you run

1. Why can a bigram distinguish `not good` from unigram counts better?
2. Why might character n-grams recognize `chagred` as related to `charged`?

Do not scroll to the output until you have an expected answer—even a rough one.

### When to use
Use n-grams when local order or noisy spelling matters and the domain vocabulary is manageable.

### When not / caution
Do not keep increasing n blindly; long phrases rarely repeat enough to estimate reliably.

### Debugging lens
Track matrix shape and sparsity as n changes—representation cost is part of the model.

In [1]:
from pathlib import Path
import re, math, json, random, statistics
import numpy as np
import pandas as pd
np.random.seed(42)
random.seed(42)
pd.set_option('display.max_colwidth', 120)
DATA = Path('data')
print('Reproducibility seed: 42')

Reproducibility seed: 42


In [2]:
from sklearn.feature_extraction.text import CountVectorizer
texts=['new york bank account','new york city travel','bank account fee','account fee refund']
vec=CountVectorizer(ngram_range=(1,2))
X=vec.fit_transform(texts)
features=np.array(vec.get_feature_names_out())
counts=np.asarray(X.sum(axis=0)).ravel()
rank=pd.DataFrame({'ngram':features,'count':counts}).sort_values(['count','ngram'],ascending=[False,True])
rank.head(12)

           ngram  count
0        account      3
1    account fee      2
2           bank      2
3   bank account      2
6            fee      2
8            new      2
9       new york      2
12          york      2
4           city      1
5    city travel      1
7     fee refund      1
10        refund      1

Bigram features can distinguish `credit card` from independent occurrences of `credit` and `card`, but the feature space grows rapidly. Character n-grams are especially useful for misspellings, morphology and language identification.

In [3]:
char=CountVectorizer(analyzer='char_wb',ngram_range=(3,5),min_df=1)
char.fit(['colour','color','colours'])
print('character n-gram features:', len(char.get_feature_names_out()))
print(char.get_feature_names_out()[:20])

character n-gram features: 27
[' co' ' col' ' colo' 'col' 'colo' 'color' 'colou' 'lor' 'lor ' 'lou'
 'lour' 'lour ' 'lours' 'olo' 'olor' 'olor ' 'olou' 'olour' 'or ' 'our']


---
    ## Production takeaways
    - Preserve preprocessing as part of the model contract; training/inference skew is an NLP failure mode, not an implementation detail.
    - Inspect intermediate representations rather than treating tokenizers/vectorizers/models as black boxes.
    - Prefer the simplest representation/model that meets quality, latency, governance, and maintenance requirements.

### What you should now be able to explain
- Explain the sparsity/expressiveness trade-off of n-grams
- Use character n-grams for robust lexical matching